# Raw → Bronze Ingestion

PySpark notebook for a Microsoft Fabric lakehouse. It reads the raw source files, keeps the rows that parse cleanly, quarantines the ones that don't, and writes Delta tables to the **bronze** layer.

| Source file | Format | Bronze table | Write mode |
|---|---|---|---|
| `customers.csv` | CSV (comma-delimited) | `bronze.bronze_customers` | overwrite |
| `accounts.csv` | CSV (pipe-delimited) | `bronze.bronze_accounts` | overwrite |
| `Transactions/<TRANSACTION_SOURCE_FILE>` | JSON lines | `bronze.bronze_transactions` | append |
| `merchants.json` | multi-line JSON | `bronze.bronze_merchants` | overwrite |
| `fx_rates.csv` | CSV | `bronze.bronze_fx_rates` | overwrite |
| `mcc_codes.csv` | CSV | `bronze.bronze_mcc_codes` | overwrite |

**Steps for each source**

1. Read with an explicit schema in `PERMISSIVE` mode, with a corrupt-record column.
2. Add ingestion metadata (source file, batch ID).
3. Split clean and corrupt records.
4. Write corrupt records as JSON to `Files/quarantine/`.
5. Write clean records to a Delta table in the `bronze` schema.
6. Write record counts to the audit table `etl_audit_log.etl_audit_logs`.

Transactions also run a schema-drift check (new or missing columns) before the read.

**Prerequisites**

- A Fabric lakehouse with the raw files under `Files/raw`.
- The `NB_FUNCTIONS` notebook, which provides `add_metadata_ingestion`, `corrupted_record` and `write_application_logs`.

**Parameters** (set in the parameters cell, can be overridden by a pipeline): `BATCH_ID`, `TRANSACTION_SOURCE_FILE`.


## Imports

In [ ]:
# Importing modules
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType
import logging

## Logger

In [ ]:
logger = logging.getLogger("Bronze Ingestion")

## Shared functions

Loads the helper functions from the `NB_FUNCTIONS` notebook.

In [ ]:
%run NB_FUNCTIONS

## Parameters

The next cell is tagged `parameters`, so a pipeline can override these values at run time.

In [ ]:
BATCH_ID = ""
TRANSACTION_SOURCE_FILE = ""

## Paths

In [ ]:
# Paths
STORAGE = "Files"
quarantine_path = f"{STORAGE}/quarantine"

# Replace with your own Fabric workspace and lakehouse IDs
WORKSPACE_ID = "<your-workspace-id>"
LAKEHOUSE_ID = "<your-lakehouse-id>"
raw_path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Files/raw"

# Alternative, with a default lakehouse attached:
# raw_path = f"{STORAGE}/raw"

## Required schemas

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS etl_audit_log;

## Customers

In [ ]:
# Column list
CUST_COLS_NAME = [
    "customer_id",
    "first_name",
    "last_name",
    "dob",
    "gender",
    "email",
    "phone",
    "pan",
    "address",
    "city",
    "state",
    "country",
    "pincode",
    "kyc_status",
    "risk_rating",
    "annual_income",
    "customer_since",
    "is_active",
    "updated_at",
    "_corrupted_record",
]

# Define schema dynamically
schema = StructType([StructField(col, StringType(), True) for col in CUST_COLS_NAME])

# File Path
path = f"{raw_path}/customers.csv"

# Read customer CSV
cust_raw_df = (
    spark.read.option("header", True)
    .option("sep", ",")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupted_record")
    .schema(schema)
    .csv(path)
    .cache()
)

In [ ]:
# Ingesting Metadata
cust_raw_df = add_metadata_ingestion(cust_raw_df, path, "customers.csv", BATCH_ID)


# Separating Clean & Corrupted Records
cust_corrupt_records = corrupted_record(cust_raw_df, "customers")
cust_clean_records = cust_raw_df.filter(F.col("_corrupted_record").isNull()).drop(
    "_corrupted_record"
)


# Sending Corrupted Records to Quarantine Folder
if cust_corrupt_records.count() > 0:
    cust_corrupt_records.write.mode("overwrite").json(
        f"{quarantine_path}/corrupt_customers/"
    )  # could use "append" with idempotent write strategy!


# Sending valid records to the Bronze layer
cust_clean_records.write.mode("overwrite").format("delta").saveAsTable(
    "bronze.bronze_customers"
)


# ===============================LOGGING=======================================

write_application_logs(
    logger_name="bronze_ingestion",
    df_name="customers",
    layer="raw",
    no_of_corr_rec=cust_corrupt_records.count(),
    no_of_cln_rec=cust_clean_records.count(),
    cln_path="Tables/bronze/bronze_customers",
    corr_path=f"{quarantine_path}/corrupted_customers",
    log_save_path="etl_audit_log.etl_audit_logs",
    batch_id=BATCH_ID,
)

## Accounts

In [ ]:
# Column list
ACC_COLS_NAME = [
    "account_id",
    "customer_id",
    "account_type",
    "account_number",
    "card_number",
    "ifsc_code",
    "branch_code",
    "currency",
    "open_date",
    "closed_date",
    "status",
    "credit_limit",
    "interest_rate",
    "updated_at",
    "_corrupted_record",
]

# Define schema dynamically
schema = StructType([StructField(col, StringType(), True) for col in ACC_COLS_NAME])

# File Path
path = f"{raw_path}/accounts.csv"

# Read Accounts CSV
acc_raw_df = (
    spark.read.option("header", True)
    .option("sep", "|")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupted_record")
    .schema(schema)
    .csv(path)
    .cache()
)

In [ ]:
# Ingesting Metadata
acc_raw_df = add_metadata_ingestion(acc_raw_df, path, "accounts.csv", BATCH_ID)


# Separating Clean & Corrupted Records
acc_corrupt_records = corrupted_record(acc_raw_df, "accounts")
acc_clean_records = acc_raw_df.filter(F.col("_corrupted_record").isNull()).drop(
    "_corrupted_record"
)


# Sending Corrupted Records to Quarantine Folder
if acc_corrupt_records.count() > 0:
    acc_corrupt_records.write.mode("overwrite").json(
        f"{quarantine_path}/corrupt_accounts/"
    )


# Sending valid records to the Bronze layer
acc_clean_records.write.mode("overwrite").format("delta").saveAsTable(
    "bronze.bronze_accounts"
)


# ===============================LOGGING=======================================

# creating schema for Logs (ONE TIME ONLY*)
# spark.sql("CREATE SCHEMA IF NOT EXISTS etl_audit_log")

write_application_logs(
    logger_name="bronze_ingestion",
    df_name="accounts",
    layer="raw",
    no_of_corr_rec=acc_corrupt_records.count(),
    no_of_cln_rec=acc_clean_records.count(),
    cln_path="Tables/bronze/bronze_accounts",
    corr_path=f"{quarantine_path}/corrupt_accounts",
    log_save_path="etl_audit_log.etl_audit_logs",
    batch_id=BATCH_ID,
)

## Transactions

In [ ]:
from pyspark.sql import functions as F

# Defining Schema
TXN_SCHEMA = """
txn_id STRING, account_id STRING, txn_ts STRING, amount STRING, currency STRING, txn_type STRING, channel STRING, status STRING,
merchant STRUCT<merchant_id: STRING, name: STRING, mcc: STRING, location: STRUCT<city: STRING, country: STRING>>,
card STRUCT<last4: STRING, present: BOOLEAN, network: STRING>,
counterparty STRUCT<name: STRING, vpa: STRING, bank_ifsc: STRING>,
tags ARRAY<STRING>, device STRUCT<ip: STRING, os: STRING>,
original_txn_id STRING, description STRING, updated_at STRING, _corrupted_record STRING
"""

# File Path
path = f"{raw_path}/Transactions/{TRANSACTION_SOURCE_FILE}"

# Read JSON with inferred schema
txn_raw_df = spark.read.json(path)

# Schema comparison
known_columns = set(txn_raw_df.columns) - {"_corrupted_record", "_corrupt_record"}

expected_columns = (set(spark.createDataFrame([], TXN_SCHEMA).columns)) - {
    "_corrupted_record"
}

new_columns = known_columns - expected_columns
missing_columns = expected_columns - known_columns

print("New columns:", new_columns)
print("Missing columns:", missing_columns)


# Route schema drift to quarantine
if missing_columns or new_columns:

    issues = [F.lit(f"SCHEMA_MISSING_COLUMN: {c}") for c in sorted(missing_columns)] + [
        F.lit(f"SCHEMA_NEW_COLUMN: {c}") for c in sorted(new_columns)
    ]

    quarantine_df = txn_raw_df.withColumn("dq_issues", F.array(*issues)).withColumn(
        "_quarantined_at", F.current_timestamp()
    )

    (
        quarantine_df.write.format("delta")
        .mode("append")
        .save("Files/quarantine/Schema_Drift/transactions")
    )

    print("Batch quarantined due to schema drift.")

else:
    print("Schema validation passed. Proceed to Read DF.")


# Reading Transaction JSON SourceFile
txn_raw_df = (
    spark.read.option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupted_record")
    .schema(TXN_SCHEMA)
    .json(f"{raw_path}/Transactions/{TRANSACTION_SOURCE_FILE}")
    .cache()
)

In [ ]:
display(txn_raw_df)

In [ ]:
# Ingesting Metadata
txn_raw_df = add_metadata_ingestion(txn_raw_df, path, TRANSACTION_SOURCE_FILE, BATCH_ID)


# Separating Clean & Corrupted Records
txn_corrupt_records = corrupted_record(txn_raw_df, "transactions")
txn_clean_records = txn_raw_df.filter(F.col("_corrupted_record").isNull()).drop(
    "_corrupted_record"
)


# Sending Corrupted Records to Quarantine Folder
if txn_corrupt_records.count() > 0:
    txn_corrupt_records.write.mode("append").json(
        f"{quarantine_path}/corrupt_transactions/"
    )


# Sending valid records to the Bronze layer
txn_clean_records.write.mode("append").format("delta").saveAsTable(
    "bronze.bronze_transactions"
)


# ===============================LOGGING=======================================

# creating schema for Logs (ONE TIME ONLY*)
# spark.sql("CREATE SCHEMA IF NOT EXISTS etl_audit_log")

write_application_logs(
    logger_name="bronze_ingestion",
    df_name="transactions",
    layer="raw",
    no_of_corr_rec=txn_corrupt_records.count(),
    no_of_cln_rec=txn_clean_records.count(),
    cln_path="Tables/bronze/bronze_transactions",
    corr_path=f"{quarantine_path}/corrupt_transactions",
    log_save_path="etl_audit_log.etl_audit_logs",
    batch_id=BATCH_ID,
)

## Merchants

In [ ]:
# Defining Merchant Schema
MERCH_SCHEMA = """
    merchant_id STRING,
    merchant_name STRING,
    category STRUCT<mcc: STRING, description: STRING>,
    aliases ARRAY<STRING>,
    address STRUCT<
        city: STRING,
        country: STRING,
        geo: STRUCT<lat: DOUBLE, lon: DOUBLE>
    >,
    risk_level STRING,
    onboarded_on STRING,
    is_active BOOLEAN,
    _corrupted_record STRING
"""

# File Path
path = f"{raw_path}/merchants.json"

# Read Merchant JSON
merch_raw_df = (
    spark.read.option("multiLine", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupted_record")
    .schema(MERCH_SCHEMA)
    .json(path)
    .cache()
)

In [ ]:
# Ingesting Metadata
merch_raw_df = add_metadata_ingestion(merch_raw_df, path, "merchants.json", BATCH_ID)


# Separating Clean & Corrupted Records
merch_corrupt_records = corrupted_record(merch_raw_df, "merchants")
merch_clean_records = merch_raw_df.filter(F.col("_corrupted_record").isNull()).drop(
    "_corrupted_record"
)


# Sending Corrupted Records to Quarantine Folder
if merch_corrupt_records.count() > 0:
    merch_corrupt_records.write.mode("overwrite").json(
        f"{quarantine_path}/corrupt_merchants/"
    )


# Sending valid records to the Bronze layer
merch_clean_records.write.mode("overwrite").format("delta").saveAsTable(
    "bronze.bronze_merchants"
)


# ===============================LOGGING=======================================

# creating schema for Logs (ONE TIME ONLY*)
# spark.sql("CREATE SCHEMA IF NOT EXISTS etl_audit_log")

write_application_logs(
    logger_name="bronze_ingestion",
    df_name="merchants",
    layer="raw",
    no_of_corr_rec=merch_corrupt_records.count(),
    no_of_cln_rec=merch_clean_records.count(),
    cln_path="Tables/bronze/bronze_merchants",
    corr_path=f"{quarantine_path}/corrupt_merchants",
    log_save_path="etl_audit_log.etl_audit_logs",
    batch_id=BATCH_ID,
)

---

## FX rates

In [ ]:
# Defining FX_Rates Schema
FX_RATES_SCHEMA = """
    rate_date STRING,
    from_ccy STRING,
    to_ccy STRING,
    rate STRING, 
    _corrupted_record STRING
"""
# File Path
path = f"{raw_path}/fx_rates.csv"

# Reading FX_RATES CSV
fx_rates_raw_df = (
    spark.read.option("header", "True")
    .option("inferSchema", "False")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupted_record")
    .schema(FX_RATES_SCHEMA)
    .csv(path)
    .cache()
)

In [ ]:
# Ingesting Metadata
fx_rates_raw_df = add_metadata_ingestion(
    fx_rates_raw_df, path, "merchants.json", BATCH_ID
)


# Separating Clean & Corrupted Records
fx_rates_corrupt_records = corrupted_record(fx_rates_raw_df, "fx_rates")
fx_rates_clean_records = fx_rates_raw_df.filter(
    F.col("_corrupted_record").isNull()
).drop("_corrupted_record")


# Sending Corrupted Records to Quarantine Folder
if fx_rates_corrupt_records.count() > 0:
    fx_rates_corrupt_records.write.mode("overwrite").json(
        f"{quarantine_path}/corrupt_fx_rates/"
    )


# Sending valid records to the Bronze layer
fx_rates_clean_records.write.mode("overwrite").format("delta").saveAsTable(
    "bronze.bronze_fx_rates"
)


# ===============================LOGGING=======================================

# creating schema for Logs (ONE TIME ONLY*)
# spark.sql("CREATE SCHEMA IF NOT EXISTS etl_audit_log")

write_application_logs(
    logger_name="bronze_ingestion",
    df_name="fx_rates",
    layer="raw",
    no_of_corr_rec=fx_rates_corrupt_records.count(),
    no_of_cln_rec=fx_rates_clean_records.count(),
    cln_path="Tables/bronze/bronze_fx_rates",
    corr_path=f"{quarantine_path}/corrupt_fx_rates",
    log_save_path="etl_audit_log.etl_audit_logs",
    batch_id=BATCH_ID,
)

---

## MCC codes

In [ ]:
# Defining MCC_CODE Schema
MCC_CODE_SCHEMA = """
    mcc STRING,
    description STRING,
    category STRING,
    is_high_risk STRING,
    _corrupted_record STRING
"""

# File Path
path = f"{raw_path}/mcc_codes.csv"

# Reading MCC_CODE CSV
mcc_codes_raw_df = (
    spark.read.option("header", "True")
    .option("inferSchema", "False")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupted_record")
    .schema(MCC_CODE_SCHEMA)
    .csv(path)
    .cache()
)

In [ ]:
# Ingesting Metadata
mcc_codes_raw_df = add_metadata_ingestion(
    mcc_codes_raw_df, path, "mcc_codes.csv", BATCH_ID
)


# Separating Clean & Corrupted Records
mcc_codes_corrupt_records = corrupted_record(mcc_codes_raw_df, "mcc_codes")
mcc_codes_clean_records = mcc_codes_raw_df.filter(
    F.col("_corrupted_record").isNull()
).drop("_corrupted_record")


# Sending Corrupted Records to Quarantine Folder
if mcc_codes_corrupt_records.count() > 0:
    mcc_codes_corrupt_records.write.mode("overwrite").json(
        f"{quarantine_path}/corrupt_mcc_codes/"
    )


# Sending valid records to the Bronze layer
mcc_codes_clean_records.write.mode("overwrite").format("delta").saveAsTable(
    "bronze.bronze_mcc_codes"
)


# ===============================LOGGING=======================================

# creating schema for Logs (ONE TIME ONLY*)
# spark.sql("CREATE SCHEMA IF NOT EXISTS etl_audit_log")

write_application_logs(
    logger_name="bronze_ingestion",
    df_name="mcc_codes",
    layer="raw",
    no_of_corr_rec=mcc_codes_corrupt_records.count(),
    no_of_cln_rec=mcc_codes_clean_records.count(),
    cln_path="Tables/bronze/bronze_mcc_codes",
    corr_path=f"{quarantine_path}/corrupt_mcc_codes",
    log_save_path="etl_audit_log.etl_audit_logs",
    batch_id=BATCH_ID,
)